In [6]:
%load_ext autoreload
%autoreload 2

# Use the density function to calculate road density, rail density, and power line density for all constituencies and all countries
The function allows the user to specify the infrastructure layer for which they want a density measure. It loops over all countries in the main dataset and calculates density and saves the calculated variables to the country shapefiles.


## Import and run the function

In [1]:
from territorial_inequality.example_sunderland import density_all

/opt/anaconda3/envs/territorial-inequality/lib/python3.11/site-packages/pyogrio/raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Point' is converted to 'Point Z'
  return ogr_read(
/opt/anaconda3/envs/territorial-inequality/lib/python3.11/site-packages/pyogrio/raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiLineString' is converted to 'MultiLineString Z'
  return ogr_read(
/opt/anaconda3/envs/territorial-inequality/lib/python3.11/site-packages/pyogrio/geopandas.py:382: UserWarning: More than one layer found in 'Africa_GIS.gdb': 'AFR_Mineral_Facilities' (default), 'AFR_Mineral_Deposits', 'AFR_Mineral_Exploration', 'AFR_Mineral_Resources_Coal', 'AFR_Mineral_Resources_Copper', 'AFR_Mineral_Resources_Gabon', 'AFR_Mineral_Resources_Mauritania', 'AFR_Mineral_Resources_PGE', 'AFR_Mineral_Resources_Potash', 'AFR_Infra_OG_Pipelines', 'AFR_Infra_Power_Stations', 'AFR_OG_Provinces_Continuous'

In [6]:
density_all(layer = "AFR_Infra_Transport_Rail")

Finished processing Algeria for layer AFR_Infra_Transport_Rail
Finished processing Angola for layer AFR_Infra_Transport_Rail
Finished processing Benin for layer AFR_Infra_Transport_Rail
Finished processing Botswana for layer AFR_Infra_Transport_Rail


DataSourceError: /Users/sophiesunderland/Desktop/CMSE802F26/Shapefiles/Shapefiles/Burkina Faso/Burkina Faso_Constituencies.shp: No such file or directory

I'm noticing that country names that are more than one word are not being read in correctly because of spacing. This interrupts the loop.
Let's see how the function worked for other countries with one word names.

In [7]:
import geopandas as gpd
country_sf = gpd.read_file(f"/Users/sophiesunderland/Desktop/CMSE802F26/Shapefiles/Shapefiles/Botswana/Botswana_Constituencies.shp")
country_sf.head()

,ISO,Country,Cons_name,road_km,road_densi,_km,_density,geometry
0,BWA,Botswana,Gaborone North,36.683494,0.382054,14.861439,0.15478,"POLYGON ((25.99297 -24.53508, 26.0208 -24.5660..."
1,BWA,Botswana,Gaborone North,36.683494,0.382054,14.861439,0.15478,"POLYGON ((25.99297 -24.53508, 26.0208 -24.5660..."
2,BWA,Botswana,Gaborone North,36.683494,0.382054,14.861439,0.15478,"POLYGON ((25.99297 -24.53508, 26.0208 -24.5660..."
3,BWA,Botswana,Gaborone North,36.683494,0.382054,14.861439,0.15478,"POLYGON ((25.99297 -24.53508, 26.0208 -24.5660..."
4,BWA,Botswana,Gaborone North,36.683494,0.382054,14.861439,0.15478,"POLYGON ((25.99297 -24.53508, 26.0208 -24.5660..."


In [8]:
country_sf[country_sf["Cons_name"] == "Gaborone South"].head()

,ISO,Country,Cons_name,road_km,road_densi,_km,_density,geometry
7,BWA,Botswana,Gaborone South,36.683494,0.382054,9.643723,0.219721,"POLYGON ((25.92917 -24.65819, 25.92921 -24.659..."
8,BWA,Botswana,Gaborone South,36.683494,0.382054,9.643723,0.219721,"POLYGON ((25.92917 -24.65819, 25.92921 -24.659..."
9,BWA,Botswana,Gaborone South,36.683494,0.382054,9.643723,0.219721,"POLYGON ((25.92917 -24.65819, 25.92921 -24.659..."
10,BWA,Botswana,Gaborone South,36.683494,0.382054,9.643723,0.219721,"POLYGON ((25.92917 -24.65819, 25.92921 -24.659..."
11,BWA,Botswana,Gaborone South,36.683494,0.382054,9.643723,0.219721,"POLYGON ((25.92917 -24.65819, 25.92921 -24.659..."


We can see that there is an error with road density where the first value for length and density was applied to all constituencies. That has since been fixed because we see that for rail, length and density are different for each constituency. This is correct, although it's strange that each constituency is repeating more than once. We want only one row per constituency.

# Use the count function to calculate counts of mineral facilities, mineral deposits, power stations, and ports for all constituencies and all countries
The function allows the user to specify the infrastructure layer for which they want a count measure. It loops over all countries in the main dataset and calculates count and saves the calculated variables to the country shapefiles.

In [3]:
from territorial_inequality.example_sunderland import count_all

In [10]:
count_all(layer = "AFR_Mineral_Facilities")

/opt/anaconda3/envs/territorial-inequality/lib/python3.11/site-packages/pyogrio/raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Point' is converted to 'Point Z'
  return ogr_read(


ValueError: Cannot set a DataFrame with multiple columns to the single column facilities

There's also an issue going on with the count variables when it comes to selecting these columns. The geometry here is much different than a length variable and I'm going to have to research more on how to do spatial calculations with 3D points.

Workshopping count loop:

In [2]:
def count_all(layer = " "):
    # Loop over all countries in main dataset 
    for country in df_all["Country"].dropna().unique():
        # create a GeoDataFrame for the selected layer
        df_sub = gpd.read_file(df_main, layer = layer)
        # filter for the current country
        df_sub = df_sub[df_sub["Country"] == country]
        # project the layer to a projected coordinate system for area calculations
        sub_proj = df_sub.to_crs("ESRI:102022")
        
        # load shapefile for current country and project to the same coordinate system
        country_sf = gpd.read_file(f"/Users/sophiesunderland/Desktop/CMSE802F26/Shapefiles/Shapefiles/{country}/{country}_Constituencies.shp")
        country_sf_proj = country_sf.to_crs("ESRI:102022")

        # intersect layer with constituencies 
        intersection = gpd.overlay(
            sub_proj,
            country_sf_proj,
            how="intersection")

        # identify the variable name based on the layer
        var = ""
        if layer == "AFR_Mineral_Facilities":
            var = "facilities"
        elif layer == "AFR_Mineral_Deposits":
            var = "deposits"
        elif layer == "AFR_Infra_Power_Stations":
            var = "stations"
        elif layer == "AFR_Infra_Transport_Ports":
            var = "ports"

        # create count variable by summing the number of points in the selected layer/variable in each constituency
        intersection[var] = intersection.groupby("Cons_name")["Cons_name"].transform("size")

        # select count and constituency name columns
        intersection_sub = intersection[[var, "Cons_name"]]
       
        # left join constituency areas with lengths to get a combined dataframe
        country_combined = country_sf_proj.merge(intersection_sub, on="Cons_name", how="left")
        
        # add necessary columns to original country shapefile        
        country_sf = country_sf.merge(
        country_combined[["Cons_name", var]],on="Cons_name",how="left")
        
        # save to the original country shapefile
        country_sf.to_file(f"/Users/sophiesunderland/Desktop/CMSE802F26/Shapefiles/Shapefiles/{country}/{country}_Constituencies.shp")
        print(f"Finished processing {country} for layer {layer}")

In [ ]:
import geopandas as gpd
df_all = gpd.read_file("/Users/sophiesunderland/Desktop/CMSE802F26/Africa_GIS/Africa_GIS.gdb")
df_main = "/Users/sophiesunderland/Desktop/CMSE802F26/Africa_GIS/Africa_GIS.gdb"
count_all(layer = "AFR_Mineral_Facilities")

In [ ]:
country_sf = gpd.read_file(f"/Users/sophiesunderland/Desktop/CMSE802F26/Shapefiles/Shapefiles/Botswana/Botswana_Constituencies.shp")
country_sf.head()

# Next steps

The code runs and the workflow is behaving as expected. There are changes that need to be made to accommodate certain country names (those that are more than one word) and to make sure that calculations are being added to shapefiles correctly.

Next steps involve refining the code to run smoothly through these issues and to fix smaller issues, such as the zero/NA difference observed in the first Milestone.

Workshopping it below before adding corrected loop as function to main script

In [1]:
def density_test(layer = " "):
    # Loop over all countries in main dataset 
    for country in df_all["Country"].dropna().unique():
        # create a GeoDataFrame for the selected layer
        df_sub = gpd.read_file(df_main, layer = layer)
        # filter for the current country
        df_sub = df_sub[df_sub["Country"] == country]
        # project the layer to a projected coordinate system for area calculations
        sub_proj = df_sub.to_crs("ESRI:102022")
        
        # load shapefile for current country and project to the same coordinate system
        country_sf = gpd.read_file(f"/Users/sophiesunderland/Desktop/CMSE802F26/Shapefiles/Shapefiles/{country}/{country}_Constituencies.shp")
        country_sf_proj = country_sf.to_crs("ESRI:102022")
        
        # calculate constituency areas in km^2
        country_sf_proj["const_area_km2"] = country_sf_proj.geometry.area / 1_000_000

        # intersect layer with constituencies 
        # intersection contains all columns in selected layer and constituency shapefile
        intersection = gpd.overlay(
            sub_proj,
            country_sf_proj,
            how="intersection")

        # identify the variable name based on the layer
        var = ""
        if layer == "AFR_Infra_Transport_Road":
            var = "road"
        elif layer == "AFR_Infra_Transport_Rail":
            var = "rail"
        elif layer == "AFR_Infra_Power_Transmission":
            var = "power"
        
        # create length variable by summing the length of the selected layer/variable in each constituency
        intersection[var] = intersection.geometry.length.groupby(intersection["Cons_name"]).sum()
        # transform from m into km
        intersection[var + "_km"] = intersection[var] / 1000

        # calculate density
        intersection[var + "_density"] = (intersection[var + "_km"] / intersection["const_area_km2"])
        
        # add necessary columns to original country shapefile        
        country_sf = country_sf.merge(
        intersection[["Cons_name", var + "_km", var + "_density"]],on="Cons_name",how="left")

        # save to the original country shapefile
        country_sf.to_file(f"/Users/sophiesunderland/Desktop/CMSE802F26/Shapefiles/Shapefiles/{country}/{country}_Constituencies.shp")
        print(f"Finished processing {country} for layer {layer}")

We know that it successfully loops over multiple countries (unless there is the error with country name) so let's check this updated function for only one country before running the entire loop again.

In [1]:
df_main = "/Users/sophiesunderland/Desktop/CMSE802F26/Africa_GIS/Africa_GIS.gdb"
def density_test(layer = " ", country = " "):
        df_sub = gpd.read_file(df_main, layer = layer)
        # filter for the current country
        df_sub = df_sub[df_sub["Country"] == country]
        # project the layer to a projected coordinate system for area calculations
        sub_proj = df_sub.to_crs("ESRI:102022")
        
        # load shapefile for current country and project to the same coordinate system
        country_sf = gpd.read_file(f"/Users/sophiesunderland/Desktop/CMSE802F26/Shapefiles/Shapefiles/{country}/{country}_Constituencies.shp")
        country_sf_proj = country_sf.to_crs("ESRI:102022")
        
        # intersect layer with constituencies 
        # intersection contains all columns in selected layer and constituency shapefile
        intersection = gpd.overlay(
            sub_proj,
            country_sf_proj,
            how="intersection")

        # identify the variable name based on the layer
        var = ""
        if layer == "AFR_Infra_Transport_Road":
            var = "road"
        elif layer == "AFR_Infra_Transport_Rail":
            var = "rail"
        elif layer == "AFR_Infra_Power_Transmission":
            var = "power"
        
        # create length variable by summing the length of the selected layer/variable in each constituency
        intersection[var] = intersection.geometry.length
        intersection = intersection.groupby("Cons_name", as_index=False)[var].sum()
        # transform from m into km
        intersection[var + "_km"] = intersection[var] / 1000

        # calculate constituency areas in km^2
        country_sf_proj["const_area_km2"] = country_sf_proj.geometry.area / 1_000_000
        # join to intersection dataframe to get the area for each constituency
        intersection = intersection.merge(country_sf_proj[["Cons_name", "const_area_km2"]], on="Cons_name", how="left")
        # calculate density
        intersection[var + "_density"] = (intersection[var + "_km"] / intersection["const_area_km2"])
        
        # add necessary columns to original country shapefile        
        country_sf = country_sf.merge(
        intersection[["Cons_name", var + "_km", var + "_density"]],on="Cons_name",how="left")

        # save to the original country shapefile
        country_sf.to_file(f"/Users/sophiesunderland/Desktop/CMSE802F26/Shapefiles/Shapefiles/{country}/{country}_Constituencies.shp")
        print(f"Finished processing {country} for layer {layer}")

In [ ]:
import geopandas as gpd
density_test(layer = "AFR_Infra_Transport_Rail", country = "Botswana")

In [ ]:
country_sf = gpd.read_file(f"/Users/sophiesunderland/Desktop/CMSE802F26/Shapefiles/Shapefiles/Botswana/Botswana_Constituencies.shp")
country_sf.head()